In [1]:
# Set up the file path
import os
os.chdir('..')

In [2]:
# Import packages
from RL4CRN.policies.parameter_generator_from_distribution import ParameterGeneratorFromDistribution
import torch
from torch.distributions import LogNormal
from RL4CRN.utils.ffnn import FFNN

In [3]:
device = 'cuda' if torch.cuda.is_available() else 'cpu'

In [4]:
# Create a continuous parameter generator with lognormal distribution
deep_layer_size = 10 * 1024
M = 100
parameter_head_attributes = {"hidden_size": 128, "num_layers": 5}
distribution = {'type': 'lognormal_1D'}
backbone_attributes={  "input_size": deep_layer_size + M, 
                        "hidden_size": parameter_head_attributes["hidden_size"], 
                        "num_layers": parameter_head_attributes["num_layers"]
                        }
parameter_generator_continuous = ParameterGeneratorFromDistribution(distribution, backbone_attributes, device=device)

In [5]:
# Generate parameters
N = 1000000
# x = torch.randn(1, deep_layer_size + M).to(device=device)  # Example input tensor of shape (N, deep_layer_size + M)
# # Repeat the input tensor N times to create a batch
# x = x.repeat(N, 1)  # shape: (N, deep_layer_size
# samples, log_probs, entropies = parameter_generator_continuous(x)

In [6]:
# # Plot the samples histogram
# import matplotlib.pyplot as plt
# plt.hist(samples.cpu().numpy(), bins=50, density=True)
# plt.title(f"Mean of samples: {samples.mean().item():.2f}, Std of samples: {samples.std().item():.2f}")
# plt.xlabel('Value')
# plt.ylabel('Density')
# plt.show()

In [7]:
backbone_attributes={  "input_size": 1024, 
                        "hidden_size": 128, 
                        "num_layers": 5
                        }
backbone = FFNN(input_size=backbone_attributes["input_size"], output_size=2, hidden_size=backbone_attributes["hidden_size"], num_layers=backbone_attributes["num_layers"]).to(device=device)  
x = torch.randn(1, backbone_attributes["input_size"]).to(device=device)  # Example input tensor of shape (1, input_size)
x = x.repeat(N, 1)  # shape: (N, input_size)
params = backbone(x) # shape: (N, 2)
means = torch.nn.functional.softplus(params[:, 0])      # shape: (N,)
stds = torch.nn.functional.softplus(params[:, 1])       # shape: (N,)
print(means[0].item(), stds[0].item())

sigma = torch.sqrt(torch.log1p(stds**2 / (means**2)))   # shape: (N,)
mu = torch.log(means) - 0.5 * sigma**2                  # shape: (N,)
dist = LogNormal(loc=mu.squeeze(-1), scale=sigma.squeeze(-1)) 
samples = dist.sample()  # shape: (N,)
print('Mean of generated samples:', samples.mean().item())
print('Std of generated samples:', samples.std().item())

0.6579245328903198 0.6581499576568604
Mean of generated samples: 0.6578264236450195
Std of generated samples: 0.6575539112091064
